# Kiel Tree-Crown Finetuning — Gesamtauswertung

Alle Modelle auf dem Kiel-Testset (kronenweise, IoU 0.5). Der Schedule ist komplett
durch; zusätzlich eine Kanal-Ablation (RGBI / +NDVI / +nDOM) bei 7.5cm.

| Modell | Trainingsdaten | Kanäle |
|---|---|---|
| `baseline` | un-finetunt (`freudenberg2022`) | 5 (RGBI+NDVI) |
| `step1_spring75` | 100% Frühjahr 7.5cm | 5 |
| `step2_spring20` | 100% Frühjahr 20cm | 5 |
| `step3_mix20` | 50/50 Sommer+Frühjahr 20cm | 5 |
| `step1_rgbi_spring75` | wie step1 | **4 (RGBI, ohne NDVI)** |
| `step1_ndom_spring75` | wie step1 | **6 (+nDOM Höhe)** |

Postprocessing ist **pro Auflösung getunt** (`pp_sweep.py`): 7.5cm = min_dist 30 / sigma 2,
20cm = min_dist 10 / sigma 3. Die Kennzahl-Tabellen in Abschnitt 2 nutzen diese getunten
Werte (autoritativ). Die Live-Matrix in 2b liest die `eval_test.csv` roh — dort kann das PP
je Datei abweichen (siehe Kopf jeder Zeile).


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
R = Path('/home/leafline/leafline/3_Model/runs')
COLORS = {'baseline':'#eda100','step1_spring75':'#2a78d6','step2_spring20':'#008300','step3_mix20':'#8a4fbe'}
def load(p, run=None):
    p=Path(p)
    if not p.exists(): print('FEHLT:',p); return None
    d=pd.read_csv(p)
    if run: d['run']=run
    return d
evals={n:load(R/f'{n}/eval_test.csv',n) for n in
       ['step1_spring75','step2_spring20','step3_mix20','step1_rgbi_spring75','step1_ndom_spring75']}
evals['baseline']=load(R/'baseline_eval.csv','baseline')
train_logs={n:load(R/f'{n}/train_log.csv') for n in
            ['step1_spring75','step2_spring20','step3_mix20','step1_rgbi_spring75','step1_ndom_spring75']}
print('geladen:', [n for n,d in evals.items() if d is not None])

## 1. Trainingsverlauf — pixelweise Val-F1 (Schedule-Modelle)

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
for run in ['step1_spring75','step2_spring20','step3_mix20']:
    tl=train_logs.get(run)
    if tl is None: continue
    ax.plot(tl['epoch'],tl['val_f1'],color=COLORS[run],lw=2,label=run)
    bi=tl['val_f1'].idxmax(); ax.scatter([tl.loc[bi,'epoch']],[tl.loc[bi,'val_f1']],color=COLORS[run],zorder=5)
    print(f"{run}: val_F1 {tl['val_f1'].max():.3f} @ep{int(tl.loc[bi,'epoch'])}")
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-F1 (pixelweise)'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Trainingsverlauf'); plt.tight_layout(); plt.show()

## 2. Kronenweise Test-F1 (getuntes PP)

### 2a. Bestes Modell je Domäne

| Domäne | bestes Modell | PP | F1 |
|---|---|---|---|
| **7.5cm** (Frühjahr) | step1_ndom (6ch) | 30/2 | **0.158** |
| **20cm** (Sommer) | step3 (50/50) | 10/3 | **0.317** (Baseline 0.340) |
| **20cm-spring** (Frühjahr) | step3 (50/50) | 10/3 | **0.144** |

### 2b. Kanal-Ablation @7.5cm (alle PP 30/2) — was jeder Eingangskanal beiträgt

| Kanäle | F1 | Precision | Recall | pred (GT=732) |
|---|---|---|---|---|
| 4 — RGBI | 0.123 | 0.166 | 0.098 | 435 |
| 5 — +NDVI (step1) | 0.150 | 0.218 | 0.115 | 385 |
| **6 — +nDOM** | **0.158** | **0.249** | 0.116 | 342 |

Jeder Kanal hilft ein wenig, **fast ausschließlich über die Precision** (weniger
Fehlalarme). NDVI trägt real bei (RGBI→+NDVI: 0.123→0.150); nDOM gibt einen kleinen
Precision-Schub (Höhe trennt Bäume von grünem Boden), pred 385→342.

### 2c. Der Recall ist die Decke — kein Hebel bewegt ihn

| Hebel | Recall @7.5cm |
|---|---|
| Postprocessing (min_dist/sigma-Sweep) | flach ~0.115 |
| Lernrate (CV 5e-5 … 2e-4) | flach / leicht schlechter |
| Kanäle 4 → 5 → 6 | 0.098 → 0.115 → 0.116 |

→ Der Engpass sind die **verpassten Kronen** selbst (kleine/niedrige Kronen, harte
IoU≥0.5-Schwelle), nicht Postprocessing, Hyperparameter oder Eingangskanäle.

### 2d. Live-Matrix aus eval_test.csv (⚠️ PP je Datei unterschiedlich)

In [ ]:
def micro(df):
    out={}
    for res,g in df.groupby('aufloesung'):
        tp,fp,fn=g.tp.sum(),g.fp.sum(),g.fn.sum()
        p=tp/(tp+fp) if tp+fp else 0.0; r=tp/(tp+fn) if tp+fn else 0.0
        out[res]=round(2*p*r/(p+r) if p+r else 0.0,3)
    return out
order=['7.5cm','20cm','20cm-spring']
tbl=pd.DataFrame({n:micro(d) for n,d in evals.items() if d is not None}).reindex(order)
print('Roh-F1 je Modell × Auflösung (eval_test.csv, gemischtes PP):')
print(tbl.to_string())
print('\nHinweis: step1/step2 eval_test.csv teils noch bei Default-PP 10/1;')
print('die getunten Heim-Werte stehen in 2a/2b.')

## 3. Diagramme

In [ ]:
# Schedule-Modelle je Auflösung (aus eval_test.csv)
runs=[r for r in ['baseline','step1_spring75','step2_spring20','step3_mix20'] if r in tbl.columns]
x=np.arange(len(order)); w=0.2
fig,ax=plt.subplots(figsize=(11,5))
for k,run in enumerate(runs):
    vals=[tbl.loc[res,run] if (res in tbl.index and not pd.isna(tbl.loc[res,run])) else 0 for res in order]
    off=(k-(len(runs)-1)/2)*w
    ax.bar(x+off,vals,width=w,label=run,color=COLORS[run])
ax.set_xticks(x); ax.set_xticklabels(order); ax.set_ylabel('F1 (roh, eval_test.csv)')
ax.set_title('Schedule-Modelle je Auflösung'); ax.legend(); ax.grid(alpha=0.3,axis='y')
plt.tight_layout(); plt.show()

# Kanal-Ablation @7.5cm (getunte 30/2-Werte, autoritativ)
ch=pd.DataFrame({'F1':[0.123,0.150,0.158],'Precision':[0.166,0.218,0.249],'Recall':[0.098,0.115,0.116]},
                index=['4 RGBI','5 +NDVI','6 +nDOM'])
ax=ch.plot(kind='bar',figsize=(8,5),color=['#2a78d6','#eda100','#008300'],rot=0)
ax.set_title('Kanal-Ablation @7.5cm (PP 30/2)'); ax.set_ylabel('Wert'); ax.grid(alpha=0.3,axis='y')
plt.tight_layout(); plt.show()

## 4. Befunde & Schedule-Stand (abgeschlossen)

**Schedule komplett:** Schritt 1 ✅ · 2 ✅ · 3 ✅ · 4 ⛔ (nicht nötig) · 4b/5 (nDOM) ✅.

1. **Getrennte Modelle je Auflösung, ein Modell je Auflösung über beide Saisons.**
   7.5cm ↔ 20cm brauchen eigene Modelle; step3 (50/50) hält bei 20cm Sommer *und*
   Frühjahr und verbessert sogar das Frühjahr ggü. step2.
2. **Kanäle:** RGBI < +NDVI < +nDOM (0.123 < 0.150 < 0.158). NDVI hilft auch im
   Frühjahr; nDOM gibt einen kleinen Precision-Schub. Beide Gewinne = Precision.
3. **Recall ist die Decke (~0.115) und robust gegen alles** (PP, LR, Kanäle). Der
   Engpass sind die verpassten (kleinen/niedrigen) Kronen — ein Daten-/Aufgaben-Thema.
4. **Wiederkehrend: Val↔Test-Umkehr.** Pixelweises Val-F1 ist kein verlässlicher
   Prädiktor der kronenweisen Test-F1 (z.B. 4ch val 0.800 > 5ch 0.788, Test umgekehrt).

**Nächste sinnvolle Richtung** (außerhalb des Schedules): gezieltes Kleinkronen-Sampling
/ mehr Daten, oder die IoU-Schwelle (0.5) für kleine Kronen überdenken.
